In [ ]:
#@title Ячейка 0 - импорт doc_map (с авто-монтированием Drive)
import sys, os, importlib

# 1. монтируем Drive, если ещё не примонтирован
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount('/content/drive')

# 2. добавляем путь к проекту
BASE = "/content/drive/MyDrive/rag_exp"
if BASE not in sys.path:
    sys.path.insert(0, BASE)

# 3. сбрасываем кэш импортов (лечит "ModuleNotFoundError" после монтирования)
importlib.invalidate_caches()

# 4. импорт
from doc_map import FILE_TO_ID
print("Загружено документов:", len(FILE_TO_ID))

Mounted at /content/drive
Загружено документов: 11


In [ ]:
#@title Ячейка 1 · Фаза 2 — расширенный грид + сравнение стратегий
# Старт: ячейки 2-13 → import rag_common → init_common. Результат: phase2_grid.csv.
# Грид по частям (append=True). 11 документов (~3,4 млн симв.), индекс из rag_01.

In [ ]:
#@title Ячейка 2 — установка зависимостей
!pip install -q -U "transformers>=4.51.0" "sentence-transformers>=2.7.0" bitsandbytes accelerate
!pip install -q scikit-learn pandas pyyaml rank_bm25 openai pymupdf
print("deps ok")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 130.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 111.8 MB/s eta 0:00:00
deps ok


In [ ]:
#@title Ячейка 3 — Drive, BASE, HF_HOME, секреты
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/rag_exp"
os.makedirs(f"{BASE}/results", exist_ok=True)
os.environ.pop("HF_HOME", None)
os.environ["HF_HOME"] = "/content/hf_cache"
os.makedirs("/content/hf_cache", exist_ok=True)
try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY"); print("OPENAI_API_KEY: ok")
except Exception:
    print("OPENAI_API_KEY: НЕ задан (S8/S9 не сработают)")
print("BASE =", BASE)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OPENAI_API_KEY: ok
BASE = /content/drive/MyDrive/rag_exp


In [ ]:
#@title Ячейка 4 — проверка GPU
import torch
assert torch.cuda.is_available(), "GPU не подключён! Среда выполнения → Сменить → T4 GPU"
print(torch.cuda.get_device_name(0),
      f"| VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


NVIDIA L4 | VRAM 23.7 GB


In [ ]:
#@title Ячейка 5 — загрузка модели Qwen3-4B-int8
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
MODEL = "Qwen/Qwen3-Embedding-4B"
tokenizer = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
bnb = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModel.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
model.eval()
print(f"VRAM после загрузки: {torch.cuda.memory_allocated()/1e9:.1f} GB")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.26k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/30.4k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

VRAM после загрузки: 4.4 GB


In [ ]:
#@title Ячейка 6 — реранкер
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512,
                        model_kwargs={"torch_dtype": torch.float16}, device="cuda")
print(f"VRAM с реранкером: {torch.cuda.memory_allocated()/1e9:.1f} GB")


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

VRAM с реранкером: 5.6 GB


In [ ]:
#@title Ячейка 7 — OpenAI клиент
oai = None
if os.environ.get("OPENAI_API_KEY"):
    try:
        from openai import OpenAI
        oai = OpenAI()
        print("OpenAI ok")
    except Exception as e:
        oai = None
        print("OpenAI клиент НЕ создан:", type(e).__name__)
else:
    print("OPENAI_API_KEY не найден — S8/S9 недоступны (в Фазе 2 не нужны, FAST без них)")

OpenAI ok


In [ ]:
#@title Ячейка 8 — импорт rag_common + init_common
import sys, importlib
sys.path.append(BASE)
importlib.invalidate_caches()
import rag_common; importlib.reload(rag_common)
from rag_common import init_common, build_or_load_index, SparseIndex, run_grid, compare_strategies

init_common(BASE, MODEL, model, tokenizer, reranker, oai)

# --- восстановление модульных переменных после reload (иначе None → cache MISS → OOM) ---
rag_common.model       = model
rag_common.tokenizer   = tokenizer
rag_common.reranker    = reranker
rag_common.MODEL       = MODEL
rag_common.BASE        = BASE
rag_common.INDEX_CACHE = f"{BASE}/index_cache"

# --- проверка, что всё на месте ---
import inspect
print("rag_common ok | tokenizer:", type(rag_common.tokenizer).__name__,
      "| model:", type(rag_common.model).__name__,
      "| reranker:", type(rag_common.reranker).__name__,
      "| oai:", "есть" if oai else "нет")
print("MODEL:", rag_common.MODEL)
print("BASE :", rag_common.BASE, "| INDEX_CACHE:", rag_common.INDEX_CACHE)
print("дедуп _ids_from:", "seen" in inspect.getsource(rag_common._ids_from))


rag_common init: MODEL=Qwen/Qwen3-Embedding-4B, INDEX_CACHE=/content/drive/MyDrive/rag_exp/index_cache, reranker=ok, oai=ok
rag_common ok | tokenizer: Qwen2Tokenizer | model: Qwen3Model | reranker: CrossEncoder | oai: есть
MODEL: Qwen/Qwen3-Embedding-4B
BASE : /content/drive/MyDrive/rag_exp | INDEX_CACHE: /content/drive/MyDrive/rag_exp/index_cache
дедуп _ids_from: True


In [ ]:
#@title Ячейка 8.5 — патч encode (батчевый)
import rag_common, torch, time
import torch.nn.functional as F

@torch.no_grad()
def encode_batched(texts, dim=None, max_length=512, batch_size=16):
    out = []
    total = len(texts)
    t0 = time.time()
    for i in range(0, total, batch_size):
        batch = texts[i:i + batch_size]
        enc = rag_common.tokenizer(batch, padding=True, truncation=True,
                                   max_length=max_length, return_tensors="pt").to(rag_common.model.device)
        hidden = rag_common.model(**enc).last_hidden_state
        vecs = rag_common.last_token_pool(hidden, enc["attention_mask"])
        if dim:
            vecs = vecs[:, :dim]
        vecs = F.normalize(vecs, p=2, dim=1)
        out.append(vecs.float().cpu())
        del enc, hidden, vecs
        torch.cuda.empty_cache()
        if (i // batch_size) % 10 == 0:
            print(f"  эмбеддинг: {i+len(batch)}/{total}  ({time.time()-t0:.0f} сек)")
    print(f"  готово: {total} за {time.time()-t0:.0f} сек")
    return torch.cat(out, dim=0)

rag_common.encode = encode_batched
print("encode:", rag_common.encode.__name__,
      "| model:", type(rag_common.model).__name__,
      "| tokenizer:", type(rag_common.tokenizer).__name__)

encode: encode_batched | model: Qwen3Model | tokenizer: Qwen2Tokenizer


In [ ]:
#@title Ячейка 9 — чтение всех PDF корпуса → real_docs
import fitz, os
CORPUS_DIR = f"{BASE}/corpus_phase1"
# FILE_TO_ID уже загружен сверху (из doc_map / автосборки) — НЕ переопределяем!
real_docs = []
for fname, doc_id in FILE_TO_ID.items():
    pdf = fitz.open(os.path.join(CORPUS_DIR, fname))
    real_docs.append({"id": doc_id, "text": "\n".join(p.get_text() for p in pdf)})
    pdf.close()
    print(f"{doc_id:22s} <- {fname}")
print("Документов:", len(real_docs))


pkps_chast_11          <- ПКПС Часть XI (Электрическое оборудование, изд.2017).pdf
pkps_chast_8           <- ПКПС Часть VIII (Системы и трубопроводы, изд.2018).pdf
rd_50_726_92           <- РД 50-726-92.pdf
gost_2_701             <- ГОСТ 2.701-2008.pdf
gost_30893_1           <- ГОСТ 30893.1-2002.pdf
gost_32569_2013        <- ГОСТ 32569-2013 (Трубопроводы технологические стальные).pdf
nd_2_020101_127_ch5    <- НД 2-020101-127 ч.5 (Правила ПОМС, навигационное оборудование).pdf
nd_2_09_006_kn6        <- НД 2-09-006 кн.6 (переиздан как 2-039901-005, 2018).pdf
nd_2_020101_174_ch2    <- НД 2-020101-174 ч.2 (Правила РС, корпус).pdf
ost_5r_4110            <- ОСТ 5Р.4110-2003 (Монтаж механизмов).pdf
ost_5_6066_75          <- ОСТ 5.6066-75 (Электромонтаж, заземление).pdf
Документов: 11


In [ ]:
#@title Ячейка 10 — базовый индекс + SparseIndex
idx_real = build_or_load_index(real_docs, dim=2048, quant="int8")
sparse_real = SparseIndex(idx_real)
print(f"Чанков: {len(idx_real['chunks'])}, key={idx_real['key']}")


[cache HIT] индекс 0724bbc94fa070d9: 7493 чанков загружено с Drive
Чанков: 7493, key=0724bbc94fa070d9


In [ ]:
#@title Ячейка 11 — QA_REAL из реестра (боевой набор)
import pandas as pd
df_reg = pd.read_csv(f"{BASE}/qa_registry.csv")
ready = df_reg[df_reg["статус"].isin(["GOOD","GOOD_OTHER_ED","GOOD_SYNTHETIC"])].copy()

# имя колонки с текстом вопроса: подстрахуемся
qcol = "вопрос_полный" if "вопрос_полный" in df_reg.columns else "вопрос"

QA_REAL = []
for _, row in ready.iterrows():
    QA_REAL.append({
        "query": str(row[qcol]),
        "relevant": {row["doc_id"]: 3},
        "Q": int(row["Q"]),
        "категория": row["категория"],
    })
print(f"QA_REAL: {len(QA_REAL)} вопросов")
print("По категориям:", ready["категория"].value_counts().to_dict())
print("По документам:", ready["doc_id"].value_counts().to_dict())


QA_REAL: 44 вопросов
По категориям: {'A': 17, 'C': 9, 'B': 9, 'D': 5, 'E': 4}
По документам: {'pkps_chast_8': 14, 'nd_2_09_006_kn6': 8, 'nd_2_020101_174_ch2': 8, 'pkps_chast_11': 3, 'rd_50_726_92': 2, 'ost_5_6066_75': 2, 'gost_2_701': 2, 'gost_30893_1': 2, 'nd_2_020101_127_ch5': 1, 'gost_32569_2013': 1, 'ost_5r_4110': 1}


In [ ]:
#@title Ячейка 11.1 · Чанковая разметка QA_REAL (Путь A, адаптация под rag_03)
# После: ячейка 11 (документный QA_REAL). Переопределяет QA_REAL → чанковый {chunk_id: 3}.
# Эталон: из примечания (номер чанка) → по пункту (глубина ≥ X.Y.Z) → fallback {doc_id: 3}.
# Сравнительные ('отлич'/'различ'/'vs') → документный уровень.
# В конце: формируется QA_DOC (всё на уровне документа) для Фазы 2A.
import re, pandas as pd

df_reg = pd.read_csv(f"{BASE}/qa_registry.csv")
good = df_reg[df_reg["статус"].isin(["GOOD","GOOD_SYNTHETIC","GOOD_OTHER_ED"])].copy()
chunks = idx_real["chunks"]; meta = idx_real["chunk_meta"]
qcol = "вопрос_полный" if "вопрос_полный" in df_reg.columns else "вопрос"

def norm(s): return re.sub(r"\s+", " ", str(s).lower()).strip()
nchunks = [norm(c) for c in chunks]
by_doc = {}
for i, m in enumerate(meta):
    by_doc.setdefault(m["doc_id"], []).append(i)

def chunk_from_note(note):
    if pd.isna(note): return None
    m = re.search(r"чанк[а-я]*\s*:?\s*(\d+)", str(note), re.I)
    return int(m.group(1)) if m else None

def clause_from_note(note):
    if pd.isna(note): return None
    m = re.search(r"п\.?\s*(\d+(?:\.\d+)+)", str(note))
    return m.group(1) if m else None

def first_clause(clause):
    if pd.isna(clause): return None
    nums = re.findall(r"\d+(?:\.\d+)+", str(clause))
    return nums[0] if nums else None

def is_comparative(clause, question):
    s = f"{clause} {question}".lower()
    return (" vs " in str(clause).lower()) or ("отлич" in s) or ("различ" in s)

def chunk_from_clause(doc_id, clause):
    if clause is None or doc_id not in by_doc: return None
    pat = re.escape(clause.strip()).replace(r"\.", r"\.\s?")
    rx = re.compile(pat)
    for i in by_doc[doc_id]:
        if rx.search(nchunks[i]): return i
    return None

QA_REAL = []
cov_note, cov_clause, cov_doc = [], [], []
for _, row in good.iterrows():
    qid, doc_id = int(row["Q"]), row["doc_id"]
    q = {"query": str(row[qcol]), "Q": qid, "категория": row["категория"], "doc_id": doc_id}
    gold, src = chunk_from_note(row.get("примечание")), None
    if gold is not None:
        src = "note"; cov_note.append(qid)
    elif is_comparative(row.get("пункт"), row[qcol]):
        gold = None
    else:
        clause = first_clause(row.get("пункт")) or clause_from_note(row.get("примечание"))
        if clause and clause.count(".") >= 2:
            gold = chunk_from_clause(doc_id, clause)
            if gold is not None: src = "clause"; cov_clause.append(qid)
    if gold is not None:
        q["relevant"], q["level"] = {gold: 3}, "chunk"
    else:
        q["relevant"], q["level"] = {doc_id: 3}, "doc"; cov_doc.append(qid)
    QA_REAL.append(q)

print(f"Всего: {len(QA_REAL)}")
print(f"  чанк из примечания: {len(cov_note)} → {cov_note}")
print(f"  чанк по пункту:     {len(cov_clause)}")
print(f"  документный:        {len(cov_doc)} → {cov_doc}")
print(f"\nИтого на уровне чанков: {len(cov_note)+len(cov_clause)} из {len(QA_REAL)}")

# === Фаза 2A: документный набор (все вопросы на уровне документа) ===
QA_DOC = [{**q, "relevant": {q["doc_id"]: 3}, "level": "doc"} for q in QA_REAL]
print(f"\nQA_DOC сформирован: {len(QA_DOC)} вопросов (документный уровень) — для Фазы 2A")



Всего: 44
  чанк из примечания: 5 → [2, 9, 10, 22, 31]
  чанк по пункту:     25
  документный:        14 → [1, 5, 6, 21, 23, 30, 36, 38, 39, 40, 42, 46, 47, 48]

Итого на уровне чанков: 30 из 44

QA_DOC сформирован: 44 вопросов (документный уровень) — для Фазы 2A


In [ ]:
import rag_common, inspect
print(inspect.getsource(rag_common.get_detailed_instruct))
print("SEARCH_TASK =", repr(rag_common.SEARCH_TASK))
print("\nsparse_search:")
print(inspect.getsource(rag_common.sparse_search))


def get_detailed_instruct(task, query):
    return f"Instruct: {task}\nQuery:{query}"

SEARCH_TASK = 'Given a question about technical standards (GOST/OST/RD), retrieve the passages that contain the answer'

sparse_search:
def sparse_search(query, sparse_index, top_k=10):
    return sparse_index.search(query, top_k=top_k)



In [ ]:
#@title Ячейка 12 — Фаза 2A: цикл по chunk_size (S6, документный уровень) → phase2a_chunksize.csv
import pandas as pd

# Документный уровень: все 44 вопроса с relevant={doc_id:3}, level="doc"
QA_DOC = [{**q, "relevant": {q["doc_id"]: 3}, "level": "doc"} for q in QA_REAL]
PHASE2A_CSV = f"{BASE}/results/phase2a_chunksize.csv"

rows_all = []
for CS in [256, 512, 1024]:
    print(f"\n=== chunk_size={CS} ===")
    idx_cs    = build_or_load_index(real_docs, chunk_size=CS, overlap=0.1, dim=2048, quant="int8")
    sparse_cs = SparseIndex(idx_cs)                      # sparse ПОД ЭТОТ размер
    print(f"  чанков: {len(idx_cs['chunks'])}, key={idx_cs['key']}")
    grid_one = {"chunk_strategy": ["fixed_512"], "chunk_size": [CS],
                "overlap": [0.1], "dim": [2048], "quant": ["int8"]}
    df_cs = run_grid(real_docs, QA_DOC, grid_one,
                     results_csv=PHASE2A_CSV, sparse_index=sparse_cs,
                     strategies=["S6"], append=(CS != 256))   # первый — перезапись, далее дозапись
    rows_all.append(df_cs)

df_2a = pd.concat(rows_all, ignore_index=True)
print("\n=== Фаза 2A: сравнение chunk_size (S6, документный уровень) ===")
print(df_2a[["chunk_size", "NDCG@5", "MRR", "Hit@5", "R@5", "search_sec"]].to_string(index=False))

# контроль: на документном уровне R@5 и NDCG@5 обязаны быть <= 1
maxr = df_2a["R@5"].max()
if maxr <= 1.0:
    print(f"\n✅ документный уровень подтверждён (max R@5 = {maxr:.3f} ≤ 1.0)")
else:
    print(f"\n⚠️ R@5 = {maxr:.3f} > 1.0 — это чанковые числа, run_grid взял не QA_DOC. Останови и проверь.")



=== chunk_size=256 ===
[cache HIT] индекс 98b0fb944c3160c6: 14980 чанков загружено с Drive
  чанков: 14980, key=98b0fb944c3160c6
Конфигураций: 1 × стратегий: 1 = 1 строк → phase2a_chunksize.csv

[cache HIT] индекс 98b0fb944c3160c6: 14980 чанков загружено с Drive
[1/1] cfg={'chunk_strategy': 'fixed_512', 'chunk_size': 256, 'overlap': 0.1, 'dim': 2048, 'quant': 'int8'}  индекс готов за 0.7s

Записано 1 строк → /content/drive/MyDrive/rag_exp/results/phase2a_chunksize.csv

=== chunk_size=512 ===
[cache HIT] индекс 0724bbc94fa070d9: 7493 чанков загружено с Drive
  чанков: 7493, key=0724bbc94fa070d9
Конфигураций: 1 × стратегий: 1 = 1 строк → phase2a_chunksize.csv

[cache HIT] индекс 0724bbc94fa070d9: 7493 чанков загружено с Drive
[1/1] cfg={'chunk_strategy': 'fixed_512', 'chunk_size': 512, 'overlap': 0.1, 'dim': 2048, 'quant': 'int8'}  индекс готов за 0.4s

Записано 1 строк → /content/drive/MyDrive/rag_exp/results/phase2a_chunksize.csv

=== chunk_size=1024 ===
[cache HIT] индекс 7d656ff63dd

In [ ]:
#@title Ячейка 12.1 — Фаза 2A: статтест 512 vs 1024 (Wilcoxon MRR + McNemar Hit@5)
import numpy as np
import rag_common
from scipy.stats import wilcoxon

def per_question(cs):
    idx = build_or_load_index(real_docs, chunk_size=cs, overlap=0.1, dim=2048, quant="int8")
    sp  = SparseIndex(idx)
    mrr_list, hit_list = [], []
    for q in QA_DOC:
        ranked = rag_common.run_search("S6", q["query"], idx,
                                       sparse_index=sp, top_k=10, dim=2048, by="doc")
        mrr_list.append(rag_common.mrr(ranked, q["relevant"]))
        hit_list.append(rag_common.hit_at_k(ranked, q["relevant"], 5))
    return np.array(mrr_list), np.array(hit_list)

mrr512, hit512 = per_question(512)
mrr1024, hit1024 = per_question(1024)

print(f"MRR  512:  mean={mrr512.mean():.4f}")
print(f"MRR 1024:  mean={mrr1024.mean():.4f}")
print(f"Hit@5  512:  {hit512.mean():.4f}  ({int(hit512.sum())}/44)")
print(f"Hit@5 1024:  {hit1024.mean():.4f}  ({int(hit1024.sum())}/44)")

# --- Wilcoxon по MRR (парный) ---
diff = mrr1024 - mrr512
nonzero = np.count_nonzero(diff)
print(f"\nWilcoxon (MRR, 1024 vs 512): ненулевых разностей {nonzero}/44")
if nonzero == 0:
    print("  все разности нулевые → различий нет, тест неприменим")
else:
    stat, p = wilcoxon(mrr1024, mrr512, zero_method="wilcox")
    print(f"  statistic={stat:.3f}  p-value={p:.4f}")
    print("  →", "значимо (p<0.05)" if p < 0.05 else "НЕ значимо (p≥0.05)")

# --- McNemar по Hit@5 (парные бинарные исходы) ---
b = int(np.sum((hit1024 == 1) & (hit512 == 0)))  # 1024 нашёл, 512 нет
c = int(np.sum((hit1024 == 0) & (hit512 == 1)))  # 512 нашёл, 1024 нет
print(f"\nMcNemar (Hit@5): только 1024={b}, только 512={c}")
if b + c == 0:
    print("  расхождений нет — оба одинаково находят, различий нет")
else:
    from scipy.stats import binomtest
    p_mc = binomtest(min(b, c), b + c, 0.5).pvalue
    print(f"  p-value (точный биномиальный)={p_mc:.4f}")
    print("  →", "значимо" if p_mc < 0.05 else "НЕ значимо")


[cache HIT] индекс 0724bbc94fa070d9: 7493 чанков загружено с Drive


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


[cache HIT] индекс 7d656ff63dd9c6c9: 3745 чанков загружено с Drive
MRR  512:  mean=0.8769
MRR 1024:  mean=0.8769
Hit@5  512:  0.9773  (43/44)
Hit@5 1024:  1.0000  (44/44)

Wilcoxon (MRR, 1024 vs 512): ненулевых разностей 6/44
  statistic=10.000  p-value=0.9156
  → НЕ значимо (p≥0.05)

McNemar (Hit@5): только 1024=1, только 512=0
  p-value (точный биномиальный)=1.0000
  → НЕ значимо


In [ ]:
#@title Ячейка 13 — Фаза 2A: итог и фиксация решения
import pandas as pd
df = pd.read_csv(f"{BASE}/results/phase2a_chunksize.csv")
print("=== Итоговая таблица Фазы 2A (S6, документный уровень) ===")
print(df[["chunk_size","NDCG@5","MRR","Hit@5","R@5","search_sec"]]
      .sort_values("chunk_size").to_string(index=False))
print("""
ВЫВОД ФАЗЫ 2A:
  • качество монотонно растёт с размером чанка (NDCG@5: 0.884→0.903→0.909)
  • 1024: Hit@5 = R@5 = 1.0 (нашёл нужный документ для всех 44)
  • статтест 512 vs 1024 (яч.12.1): Wilcoxon p=0.92, McNemar p=1.0 → различий НЕТ
  • 1024 даёт вдвое меньше чанков (3745 vs 7493) → меньше VRAM, быстрее
РЕШЕНИЕ: chunk_size = 1024 для Фазы 2B (равное качество + лучшая эффективность)
""")
